In [23]:
import base64
import numpy as np
from io import BytesIO
from PIL import Image as PILImage
from tf_keras.models import load_model
from google.colab.output import eval_js
from IPython.display import display, Javascript

def video_stream():
  """Sets up the video window in your browser."""
  js = Javascript('''
    var video; var div = null; var stream; var captureCanvas; var labelElement;
    var pendingResolve = null; var shutdown = false;

    function removeDom() {
       stream.getTracks().forEach(track => track.stop());
       div.remove(); div = null;
    }

    function onAnimationFrame() {
      if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 224, 224);
          result = captureCanvas.toDataURL('image/jpeg', 0.8);
        }
        var lp = pendingResolve; pendingResolve = null; lp(result);
      }
    }

    async function createDom() {
      div = document.createElement('div');
      div.style.border = '2px solid black'; div.style.padding = '10px';
      div.style.width = '250px'; div.style.background = 'white';

      video = document.createElement('video');
      video.style.display = 'block'; video.width = 224; video.height = 224;
      video.autoplay = true; video.playsInline = true;

      stream = await navigator.mediaDevices.getUserMedia({video: {width: 224, height: 224}});
      div.appendChild(video);
      video.srcObject = stream;

      labelElement = document.createElement('div');
      labelElement.style.fontSize = '18px'; labelElement.style.fontWeight = 'bold';
      labelElement.style.marginTop = '10px'; labelElement.innerText = "Initializing...";
      div.appendChild(labelElement);

      document.body.appendChild(div);
      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 224; captureCanvas.height = 224;
      window.requestAnimationFrame(onAnimationFrame);
    }

    async function streamFrame(label) {
      if (shutdown) return "";
      if (div === null) await createDom();
      labelElement.innerText = label;
      return new Promise(resolve => { pendingResolve = resolve; });
    }
    ''')
  display(js)

def video_frame(label):
  return eval_js('streamFrame("{}")'.format(label))

In [24]:
try:
    model = load_model("/content/keras_model.h5", compile=False)
    class_names = open("//content/labels.txt", "r").readlines()
    print("AI Brain Loaded!")
except Exception as e:
    print(f"Make sure keras_model.h5 and labels.txt are uploaded! Error: {e}")

video_stream()
label = 'Waiting for camera...'

try:
    while True:
        js_reply = video_frame(label)
        if not js_reply: break

        header, encoded = js_reply.split(",", 1)
        binary = base64.b64decode(encoded)
        image = PILImage.open(BytesIO(binary)).convert("RGB")
        image_np = np.asarray(image, dtype=np.float32)
        normalized_image = (image_np.reshape(1, 224, 224, 3) / 127.5) - 1

        prediction = model.predict(normalized_image, verbose=0)
        index = np.argmax(prediction)
        conf = prediction[0][index] * 100

        label = f"{class_names[index].strip()} ({conf:.1f}%)"

except KeyboardInterrupt:
    print("Camera stopped.")
finally:
    eval_js('shutdown = true; removeDom()')

AI Brain Loaded!


<IPython.core.display.Javascript object>

Camera stopped.
